# Monotonic effects (`mo()`)

When a predictor is **ordered** but the distance between adjacent levels is not
meaningful — e.g. income brackets, Likert items, education level — neither of the
two obvious encodings works well:

- treating it as numeric forces the effect to be linear in the codes
  ($0, 1, 2, \dots$), which is rarely defensible;
- treating it as a regular categorical predictor ignores the ordering and uses
  $K-1$ independent dummy coefficients.

Bambi's `mo()` term, ported from the R package
[brms](https://paulbuerkner.com/brms/) following Bürkner & Charpentier (2020),
splits the difference. The contribution of a monotonic predictor with $K$ levels
to the linear predictor is

$$
\eta_n \;=\; b \cdot D \cdot \sum_{i=1}^{x_n} \zeta_i,
$$

where:

- $b$ is a scalar **slope** (any real value);
- $\zeta = (\zeta_1, \ldots, \zeta_D)$ is a length-$D$ **simplex**
  ($\zeta_i \geq 0,\ \sum_i \zeta_i = 1$) with $D = K - 1$, representing the
  *relative* size of each step from one level to the next;
- $D$ rescales the cumulative sum so that the contribution at the highest level
  is exactly $b$.

The simplex carries a Dirichlet prior; the default is
$\zeta \sim \text{Dirichlet}(1, \ldots, 1)$ (uniform on the simplex).

This notebook walks through:

1. A standalone `mo(x)` main effect on an ordered categorical predictor.
2. Why treating that predictor as numeric or as a regular categorical is worse.
3. The `id=` shared-simplex mechanism for conditional monotonicity.
4. `mo()` in an interaction, e.g. `mo(x) * z`.
5. Group-specific monotonic effects, `(mo(x) | g)`.

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bambi as bmb

rng = np.random.default_rng(2026)
az.style.use("arviz-darkgrid")

## 1. A standalone `mo()` main effect

We simulate the life-satisfaction example from the brms vignette: an ordered
`income` factor with four brackets, and a continuous outcome `ls` whose mean
across brackets is **30, 60, 70, 75** — clearly monotonic but **not linear** in
the level codes (the jumps are 30, 10, 5).

In [ ]:
levels = ["below_20", "20_to_40", "40_to_100", "greater_100"]
true_means = {"below_20": 30.0, "20_to_40": 60.0, "40_to_100": 70.0, "greater_100": 75.0}

n = 400
income = pd.Categorical(rng.choice(levels, n), categories=levels, ordered=True)
ls = np.array([true_means[i] for i in income]) + rng.normal(0, 7, n)
df = pd.DataFrame({"ls": ls, "income": income})
df.head()

In [ ]:
model_mo = bmb.Model("ls ~ mo(income)", df)
model_mo

In [ ]:
idata_mo = model_mo.fit(
    draws=1000, tune=1000, chains=4, random_seed=20260522, progressbar=False
)
az.summary(idata_mo, var_names=["Intercept", "mo(income)_b", "mo(income)_simplex", "sigma"])

The simplex posterior recovers the unequal spacing of the category means: the
first jump is the largest (~0.67 of the total range), the second moderate
(~0.22), the third smallest (~0.11). The slope `b` lands at the per-step average
increase, about $(75 - 30)/3 = 15$.

We can reconstruct the per-category fitted means and check against truth:

In [ ]:
new_df = pd.DataFrame(
    {"income": pd.Categorical(levels, categories=levels, ordered=True)}
)
preds = model_mo.predict(idata_mo, kind="response_params", data=new_df, inplace=False)
fitted = preds.posterior["mu"].mean(("chain", "draw")).to_numpy()
truth = np.array([true_means[l] for l in levels])

pd.DataFrame({"level": levels, "true_mean": truth, "fitted_mean": fitted.round(2)})

## 2. Why neither numeric nor categorical encoding works as well

Fit two alternative models on the same data:

- `ls ~ income_code` treats the four levels as evenly spaced numbers $0, 1, 2, 3$;
- `ls ~ C(income)` uses three independent dummy coefficients.

We compare their leave-one-out information criteria with the `mo()` model. The
categorical model is the most flexible and `mo()` is a constrained version of
it; the numeric model is most restrictive.

In [ ]:
df["income_code"] = pd.Categorical(df["income"]).codes.astype(float)

model_num = bmb.Model("ls ~ income_code", df)
idata_num = model_num.fit(
    draws=1000, tune=1000, chains=4, random_seed=20260522, progressbar=False
)

model_cat = bmb.Model("ls ~ C(income)", df)
idata_cat = model_cat.fit(
    draws=1000, tune=1000, chains=4, random_seed=20260522, progressbar=False
)

az.compare(
    {"numeric": idata_num, "categorical": idata_cat, "monotonic": idata_mo},
    ic="loo",
)

The categorical and `mo()` models are statistically indistinguishable on this
dataset (both fit the four category means well), but `mo()` uses just **two**
parameters where the categorical model uses **three** — and the savings grow as
$K$ grows. The numeric encoding lags behind because it cannot represent the
uneven spacing.

## 3. Shared simplex via `id=` (conditional monotonicity)

When two `mo()` terms involve predictors that you believe share the same shape
of monotonic effect, you can tie their simplices with `id=`. This is brms's
*conditional monotonicity* mechanism (Bürkner & Charpentier 2020, §3): only a
single Dirichlet variable is sampled, and both terms multiply it by their own
slope.

Two ordered predictors built from the same template:

In [ ]:
shape = np.array([0.0, 30.0, 40.0, 45.0])  # cumulative effect at each code
income1 = pd.Categorical(rng.choice(levels, n), categories=levels, ordered=True)
income2 = pd.Categorical(rng.choice(levels, n), categories=levels, ordered=True)
mu = 5.0 + shape[income1.codes] + shape[income2.codes]
y = mu + rng.normal(0, 5, n)
df_id = pd.DataFrame({"y": y, "income1": income1, "income2": income2})

model_id = bmb.Model(
    "y ~ mo(income1, id='shape') + mo(income2, id='shape')", df_id
)
idata_id = model_id.fit(
    draws=1000, tune=1000, chains=4, random_seed=20260522, progressbar=False
)
az.summary(idata_id, var_names=["simplex_shape"])

Only one `simplex_shape` variable is in the posterior — both terms reuse it.
Each term still has its own slope (`mo(income1, id='shape')_b`,
`mo(income2, id='shape')_b`).

## 4. Interactions with `mo()`

`mo()` interacts cleanly with continuous and categorical predictors. The model
below has an additive monotonic effect of `income` plus an interaction with a
continuous `x`:

In [ ]:
x = rng.normal(size=n)
mu = 5.0 + shape[income.codes] + 4.0 * x + 0.2 * shape[income.codes] * x
y = mu + rng.normal(0, 3, n)
df_int = pd.DataFrame({"y": y, "income": income, "x": x})

model_int = bmb.Model("y ~ mo(income) * x", df_int)
idata_int = model_int.fit(
    draws=1000, tune=1000, chains=4, random_seed=20260522, progressbar=False
)
az.summary(
    idata_int,
    var_names=["Intercept", "mo(income)_b", "x", "mo(income):x_b"],
)

## 5. Group-specific monotonic effects, `(mo(x) | g)`

When the monotonic slope varies by group, write it just like any other
group-specific term. The simplex is shared across groups (matching brms); only
the slope $b$ has a per-group offset $r_g \sim \mathcal{N}(0, \sigma_g)$ with
$\sigma_g \sim \text{HalfNormal}$.

In [ ]:
g_levels = [f"g{i}" for i in range(1, 6)]
slope_per_group = {"g1": 1.0, "g2": 1.3, "g3": 0.7, "g4": 1.5, "g5": 0.5}

group = pd.Categorical(rng.choice(g_levels, n), categories=g_levels)
income_gs = pd.Categorical(rng.choice(levels, n), categories=levels, ordered=True)
g_mult = np.array([slope_per_group[g] for g in group])
mu = 5.0 + shape[income_gs.codes] * g_mult
y = mu + rng.normal(0, 3, n)
df_gs = pd.DataFrame({"y": y, "income": income_gs, "g": group})

model_gs = bmb.Model(
    "y ~ mo(income, id='inc') + (mo(income, id='inc') | g)", df_gs
)
idata_gs = model_gs.fit(
    draws=1000, tune=1000, chains=4, random_seed=20260522,
    target_accept=0.95, progressbar=False,
)
az.summary(
    idata_gs,
    var_names=[
        "mo(income, id='inc')_b",
        "mo(income, id='inc')|g_sigma",
        "mo(income, id='inc')|g",
        "simplex_inc",
    ],
)

Using `id='inc'` ties the simplex used in the main effect to the one used in
the group-specific term — matching the default brms convention where the
monotonic *shape* is shared across groups while only the *magnitude* varies.

## References

- Bürkner, P.-C., & Charpentier, E. (2020). Modelling monotonic effects of
  ordinal predictors in Bayesian regression models. *British Journal of
  Mathematical and Statistical Psychology*, 73(3), 420-451.
  [doi:10.1111/bmsp.12195](https://doi.org/10.1111/bmsp.12195)
- brms `mo()` reference:
  [`paulbuerkner.com/brms/reference/mo.html`](https://paulbuerkner.com/brms/reference/mo.html)

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w